# Recuperação de Informação (SCC0282)

Busca lexical, semântica, híbrida (RRF) e com reranking, mais RAG, no NFCorpus.

Este notebook executa os scripts `00` a `08` em ordem. Roda no Colab ou localmente, GPU só é necessária para a LLM local do RAG.

## 0. Ambiente

In [ ]:
# %cd corpus-hybrid-rag
!pip install -q -r requirements.txt
import nltk; nltk.download('stopwords', quiet=True)

## 1. Dados

In [ ]:
!python scripts/00_download_data.py

## 2. Ajuste de parâmetros (dev)

In [ ]:
!python scripts/01_dev_tuning.py --stage all

## 3. Avaliação no test

In [ ]:
!python scripts/02_run_test_pipeline.py

In [ ]:
import pandas as pd
pd.read_csv('results/summary_test.csv')

## 4. Erros ortográficos

In [ ]:
!python scripts/03_typo_experiment.py

## 5. Análise por consulta

In [ ]:
!python scripts/04_analysis_cases.py > /dev/null
print(open('results/analysis/lex_vs_sem_cases.md').read()[:3000])

## 6. RAG
### 6.1 Contexto e prompts

In [ ]:
!python scripts/05_rag_prepare.py

### 6.2 Geração
Execução entregue: qwen2.5:7b local via Ollama (`ollama pull qwen2.5:7b`), com respostas em cache em `artifacts/llm_cache.jsonl`.

In [ ]:
!python scripts/06_rag_generate.py --backend openai --base-url http://localhost:11434/v1 --model qwen2.5:7b

### 6.3 Avaliação humana
As planilhas já estão preenchidas em `results/rag_eval_*.csv`.

In [ ]:
# !python scripts/07_rag_evaluate.py make-sheet

In [ ]:
!python scripts/07_rag_evaluate.py merge --final results/rag_eval_amanda.csv --sheets results/rag_eval_amanda.csv results/rag_eval_leonardo.csv results/rag_eval_avaliador_ia.csv

## 7. Testes e relatório

In [ ]:
!python -m pytest -q tests

In [ ]:
!python scripts/08_make_report_assets.py